# Aula 6 — Quantum ESPRESSO on Google Colab

Hands-on: silicon SCF, convergence tests (`ecutwfc`, k mesh) and the lattice parameter.
Run the cells in order. Every number quoted below was checked with QE 6.7 (the version `apt` installs on Colab).

## 1. Install Quantum ESPRESSO
About 1 min. Re-run this cell every time the Colab session restarts (your files on Drive stay). The last line should print `/usr/bin/pw.x`.

In [ ]:
# Quantum ESPRESSO from Ubuntu's package manager (about 1 min)
!apt-get update -qq
!apt-get install -y -qq quantum-espresso > /dev/null
!which pw.x

## 2. Get the course material into your Google Drive
The Colab machine is wiped when the session ends; Google Drive is not. The clone is skipped if the folder already exists, so this cell is safe to re-run in every session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")                       # authorize when asked
%cd /content/drive/MyDrive
REPO = "https://github.com/flavianojs/dft-cbpf.git"
![ -d dft-cbpf ] || git clone -q --depth 1 {REPO}
%cd dft-cbpf/calculations

## 3. First run by hand (Terminal)
Open the **Terminal** (button at the bottom left of Colab) and type:

```bash
cd /content/drive/MyDrive/dft-cbpf/calculations
mkdir -p aula6
cp ../src/silicon/pw.scf.silicon.in aula6/
```

In the file panel (folder icon on the left) go to drive › MyDrive › dft-cbpf › calculations › aula6, open `pw.scf.silicon.in` and change one line, then save (Ctrl+S):

```
pseudo_dir = '../pseudos/'        ->   pseudo_dir = '../../src/pseudos'
```

Back in the Terminal:

```bash
cd aula6
pw.x < pw.scf.silicon.in > pw.scf.silicon.out
grep "total energy *=" pw.scf.silicon.out
```

Expected: four iterations, the last one `!    total energy = -15.85039422 Ry` (same as the reference output in `calculations/reference/`).

## 4. Automate: four helper functions
What we did by hand (copy, edit one line, run, grep) becomes four functions, so we can run dozens of calculations. Each run goes to `calculations/aula6/`.

In [ ]:
import re, subprocess
from pathlib import Path

def set_var(text, name, value):        # name = value in a namelist
    pat = rf"^(\s*{re.escape(name)}\s*=\s*).*$"
    text, n = re.subn(pat, rf"\g<1>{value}", text, count=1, flags=re.M | re.I)
    assert n == 1, f"{name} not found"
    return text

def set_kpts(text, mesh):              # the line after K_POINTS
    return re.sub(r"(K_POINTS.*\n).*", rf"\g<1>  {mesh}", text, count=1)

def run(text, name, workdir="aula6"):  # write name.in, run pw.x, return output
    w = Path(workdir); w.mkdir(exist_ok=True)
    (w / f"{name}.in").write_text(text)
    with open(w / f"{name}.out", "w") as out:
        subprocess.run(["pw.x", "-in", f"{name}.in"], cwd=w, stdout=out)
    return (w / f"{name}.out").read_text()

def etot(out):                         # last "!    total energy", in Ry
    return float(re.findall(r"!\s+total energy\s+=\s+(\S+)", out)[-1])

## 5. The same SCF from Python
Expected: same four iterations and `-15.85039422 Ry` as the run by hand. A new name (`si_scf`) keeps the hand-made output.

In [ ]:
base = Path("../src/silicon/pw.scf.silicon.in").read_text()
inp = set_var(base, "pseudo_dir", "'../../src/pseudos'")   # relative to aula6/
out = run(inp, "si_scf")
print(out.splitlines()[1].strip())
!grep "total energy *=" aula6/si_scf.out

## 6. Convergence with `ecutwfc` (k mesh fixed at 6×6×6, centered)
Expected: 30 Ry is 8.4 meV/atom above 60 Ry; 1 meV/atom is reached between 40 and 48 Ry.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
RY_MEV = 13605.693                                  # 1 Ry in meV

ecuts = [12, 16, 20, 24, 28, 32, 36, 40, 48, 60]
E = np.array([etot(run(set_var(inp, "ecutwfc", ec), f"ecut{ec}")) for ec in ecuts])
dE = (E - E[-1]) / 2 * RY_MEV                       # meV/atom (2 atoms), vs 60 Ry
plt.semilogy(ecuts[:-1], dE[:-1], "o-")
plt.axhline(1, ls="--"); plt.xlabel("ecutwfc (Ry)"); plt.ylabel("E - E(60 Ry) (meV/atom)")
plt.show()

## 7. Convergence with the k mesh (`ecutwfc` = 30 Ry)
Expected: centered 6×6×6 → 12 meV/atom; shifted 6×6×6 → 0.04 meV/atom. The 16×16×16 reference takes ~20 s.

In [ ]:
ks = [2, 4, 6, 8, 10, 12]
E_ref = etot(run(set_kpts(inp, "16 16 16 1 1 1"), "k16s1"))   # reference
for s in (0, 1):                                              # 0: centered, 1: shifted
    E = np.array([etot(run(set_kpts(inp, f"{n} {n} {n} {s} {s} {s}"), f"k{n}s{s}")) for n in ks])
    plt.semilogy(ks, abs(E - E_ref) / 2 * RY_MEV, "o-", label=f"shift {s}")
plt.axhline(1, ls="--"); plt.legend(); plt.xlabel("n (n x n x n mesh)"); plt.ylabel("meV/atom")
plt.show()

## 8. Lattice parameter (40 Ry, shifted 8×8×8)
Expected: `a0 = 10.208 bohr = 5.402 Å` (experiment: 5.431 Å).

In [ ]:
inp_a = set_kpts(set_var(inp, "ecutwfc", 40), "8 8 8 1 1 1")
alats = np.arange(9.9, 10.61, 0.1).round(2)
E = np.array([etot(run(set_var(inp_a, "celldm(1)", a), f"a{a:.2f}")) for a in alats])
c = np.polyfit(alats, E, 3)                                   # cubic fit of E(a)
r = np.roots(np.polyder(c)).real
a0 = r[np.argmin(abs(r - 10.2))]                              # the minimum
print(f"a0 = {a0:.3f} bohr = {a0 * 0.529177:.3f} Å")
plt.plot(alats, (E - E.min()) / 2 * RY_MEV, "o")
x = np.linspace(9.85, 10.65, 200); plt.plot(x, (np.polyval(c, x) - E.min()) / 2 * RY_MEV)
plt.axvline(a0, ls=":"); plt.axvline(10.263, ls="--", c="C1")
plt.xlabel("celldm(1) (bohr)"); plt.ylabel("E - E_min (meV/atom)"); plt.show()

## Extras
### vc-relax
`relax` and `vc-relax` need the `&IONS` and `&CELL` namelists, even if empty. Expected: `CELL_PARAMETERS` with 0.497332426 instead of 0.5 → a = 10.26 × 0.497332/0.5 = 10.205 bohr.

In [ ]:
vc = set_var(inp_a, "calculation", "'vc-relax'")
vc = vc.replace("ATOMIC_SPECIES", "&IONS\n/\n&CELL\n/\nATOMIC_SPECIES")   # relax needs both
out = run(vc, "vcrelax")
!grep -A6 "Begin final coordinates" aula6/vcrelax.out

### Number of plane waves at Γ: 30 vs 60 Ry
Does it grow like 2^(3/2) ≈ 2.8?

In [ ]:
!grep -m1 "PWs)" aula6/pw.scf.silicon.out aula6/ecut60.out

### Next session
Re-run sections 1 and 2 (QE is reinstalled, the clone is skipped), then the cells of sections 4 and 5 to define the functions and `inp` again. Everything in `aula6/` is still in your Drive.